# ARC-AGI-3 Solo — Submission v1 (graph exploration)

Phase 1 baseline. Frame-graph exploration agent ported from the dolphin-in-a-coma `just-explore` 3rd-place strategy: hash every frame, treat actions as edges, BFS to the nearest open frontier with a 5-group priority over action candidates.

Source modules live under `src/arc_agi3_solo/` in the repo and are mirrored inline here so the notebook is self-contained on Kaggle.

License of ported code: MIT (Rudakov 2025, see vendor/just-explore/LICENSE).

## 1. Install vendored wheels

In [ ]:
import subprocess, sys, glob, os

WHEEL_DIR = "/kaggle/input/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"
wheels = sorted(glob.glob(os.path.join(WHEEL_DIR, "*.whl")))
print(f"Found {len(wheels)} wheels in {WHEEL_DIR}")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
    "--find-links", WHEEL_DIR, "arc-agi", "arcengine",
])
print("install ok")


## 2. Force COMPETITION (offline) mode

In [ ]:
import os

os.environ["OPERATION_MODE"] = "competition"
os.environ["ENVIRONMENTS_DIR"] = "/kaggle/input/arc-prize-2026-arc-agi-3/environment_files"

from arc_agi import Arcade
from arcengine import FrameData, FrameDataRaw, GameAction, GameState
print("arc_agi imported, mode =", os.environ["OPERATION_MODE"])


## 3. Inlined source: Agent base, GraphExplorer, FrameProcessor, GraphAgent

In [ ]:
import logging, time
from abc import ABC, abstractmethod
from typing import Optional
from arc_agi import EnvironmentWrapper

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger()

class Agent(ABC):
    MAX_ACTIONS = 5000
    def __init__(self, card_id, game_id, agent_name, arc_env, tags=None, ROOT_URL="", record=False):
        self.card_id = card_id
        self.game_id = game_id
        self.agent_name = agent_name
        self.arc_env = arc_env
        self.tags = tags or []
        self.frames = [FrameData(levels_completed=0)]
        self.action_counter = 0
        self.timer = 0.0
        self.guid = ""
    @property
    def name(self): return f"{self.game_id}.{self.__class__.__name__.lower()}"
    @property
    def state(self): return self.frames[-1].state
    @property
    def levels_completed(self): return int(self.frames[-1].levels_completed)
    @property
    def seconds(self): return round(time.time() - self.timer, 2)
    @property
    def fps(self):
        if self.action_counter == 0: return 0.0
        return round(self.action_counter / max(self.seconds, 0.1), 2)
    def main(self):
        self.timer = time.time()
        while not self.is_done(self.frames, self.frames[-1]) and self.action_counter <= self.MAX_ACTIONS:
            action = self.choose_action(self.frames, self.frames[-1])
            frame = self._step(action)
            if frame is not None:
                self._append(frame)
            self.action_counter += 1
    def _step(self, action):
        try:
            data = action.action_data.model_dump()
            raw = self.arc_env.step(action, data=data, reasoning=data.get("reasoning", {}))
        except Exception:
            log.exception("step failed for %s", action.name)
            return None
        if raw is None: return None
        return FrameData(
            game_id=raw.game_id, frame=[a.tolist() for a in raw.frame], state=raw.state,
            levels_completed=raw.levels_completed, win_levels=raw.win_levels,
            guid=raw.guid, full_reset=raw.full_reset, available_actions=raw.available_actions,
        )
    def _append(self, frame):
        self.frames.append(frame)
        if frame.guid:
            self.guid = frame.guid
    @abstractmethod
    def is_done(self, frames, latest_frame): ...
    @abstractmethod
    def choose_action(self, frames, latest_frame): ...


In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Dict, Hashable, List, Optional, Set, Tuple

import numpy as np
import random

INFINITY = np.iinfo(np.int32).max

# Per-edge metadata. Group: priority bucket. Result: 1 success, -1 fail, 0 untested.
# Target: name of destination node (empty until tested). Distance: BFS distance
# from the frontier (maintained by GraphExplorer). Errors: probe-error count.
edge_dtype = np.dtype([
    ("group", "i4"),
    ("result", "i4"),
    ("target", "U32"),
    ("distance", "i4"),
    ("errors", "i4"),
])


@dataclass
class NodeInfo:
    name: Hashable
    total_candidates: int
    num_groups: int = 1
    active_group: int = 0
    group2remaining_candidate_ids: List[Set[int]] = field(default_factory=list)
    edge_data: np.ndarray = field(default_factory=lambda: np.empty(0, dtype=edge_dtype))
    error_threshold: int = 3
    closed: bool = False
    distance: Optional[float] = 0

    def __post_init__(self) -> None:
        assert self.name is not None, "Node name must be provided"
        if self.num_groups > 1 and self.group2remaining_candidate_ids is None:
            raise ValueError("group2remaining_candidate_ids required when num_groups > 1")
        if self.num_groups == 1 and not self.group2remaining_candidate_ids:
            self.group2remaining_candidate_ids = [set(range(self.total_candidates))]
        self.group2remaining_candidate_ids = [set(s) for s in self.group2remaining_candidate_ids]
        self.edge_data = np.zeros(self.total_candidates, dtype=edge_dtype)
        for gid, remaining in enumerate(self.group2remaining_candidate_ids):
            self.edge_data["group"][list(remaining)] = gid

    @property
    def has_open(self) -> bool:
        return bool((self.edge_data["result"] == 0).any())

    def has_open_group(self, group_id: int) -> bool:
        for gid in range(group_id + 1):
            if self.group2remaining_candidate_ids[gid]:
                return True
        return False

    def record_test(self, edge_idx: int, success: int, target_node: Optional[Hashable] = None) -> bool:
        edge_group_id = self.edge_data[edge_idx]["group"]
        assert (
            self.edge_data["result"][edge_idx] == 0
            and self.edge_data["target"][edge_idx] == ""
            and self.edge_data["distance"][edge_idx] == 0
        ), "edge must be untested before recording"

        if success == -1:
            self.edge_data["errors"][edge_idx] += 1
            if self.edge_data["errors"][edge_idx] >= self.error_threshold:
                self.edge_data["errors"][edge_idx] = 0
                new_group_id = edge_group_id + 1
                if new_group_id > self.num_groups - 1:
                    self.group2remaining_candidate_ids[edge_group_id].discard(edge_idx)
                    self.edge_data["result"][edge_idx] = -1
                    self.edge_data["distance"][edge_idx] = INFINITY
                    return True
                self.edge_data["group"][edge_idx] = new_group_id
                self.group2remaining_candidate_ids[new_group_id].add(edge_idx)
                self.group2remaining_candidate_ids[edge_group_id].discard(edge_idx)
            return False

        self.group2remaining_candidate_ids[edge_group_id].discard(edge_idx)
        if success == 1:
            self.edge_data["target"][edge_idx] = str(target_node)
            self.edge_data["distance"][edge_idx] = -1
            self.edge_data["result"][edge_idx] = 1
        elif success == 0:
            self.edge_data["distance"][edge_idx] = INFINITY
            self.edge_data["result"][edge_idx] = -1
        return True


class GraphExplorer:
    def __init__(
        self,
        start_node: Optional[Hashable] = None,
        num_candidates: Optional[int] = None,
        group2remaining_candidate_ids: Optional[List[Set[int]]] = None,
        n_groups: int = 1,
        verbose_level: int = 0,
    ) -> None:
        self._verbose_level = verbose_level
        self._n_groups = max(1, n_groups)
        self.reset()

    def reset(self) -> None:
        self._nodes: Dict[Hashable, NodeInfo] = {}
        self._G: Dict[Hashable, Set[Tuple[int, Hashable]]] = defaultdict(set)
        self._G_rev: Dict[Hashable, Set[Tuple[int, Hashable]]] = defaultdict(set)
        self._frontier: Set[Hashable] = set()
        self._dist: Dict[Hashable, int] = {}
        self._next: Dict[Hashable, Tuple[int, Hashable]] = {}
        self._active_group: int = 0
        self.suspicious_transitions: Dict[Tuple[Hashable, int, Hashable], int] = {}
        self.suspicious_transitions_threshold: int = 3
        self._empty = True

    def initialize(
        self,
        start_node: Optional[Hashable] = None,
        num_candidates: Optional[int] = None,
        group2remaining_candidate_ids: Optional[List[Set[int]]] = None,
    ) -> None:
        if start_node is not None:
            self._add_new_node(start_node, num_candidates, group2remaining_candidate_ids=group2remaining_candidate_ids)

    def record_test(
        self,
        node: Hashable,
        edge_idx: Hashable,
        success: bool,
        target_node: Optional[Hashable] = None,
        target_num_candidates: Optional[int] = None,
        group2remaining_candidate_ids: Optional[List[Set[int]]] = None,
        suspicious_transition: bool = False,
    ) -> None:
        if node not in self._nodes:
            raise KeyError(f"unknown node {node!r}")
        node_info = self._nodes[node]

        if node_info.closed:
            if target_node == self._nodes[node].edge_data["target"][edge_idx]:
                return
            dist_to_frontier = self._dist.get(target_node, 0)
            prev_target_node = self._nodes[node].edge_data["target"][edge_idx]
            prev_dist_to_frontier = self._dist.get(prev_target_node, INFINITY)
            if dist_to_frontier >= prev_dist_to_frontier:
                return

        if suspicious_transition:
            key = (node, edge_idx, target_node)
            self.suspicious_transitions[key] = self.suspicious_transitions.get(key, 0) + 1
            if self.suspicious_transitions[key] < self.suspicious_transitions_threshold:
                return

        node_info.record_test(edge_idx, success, target_node)

        if success == 1:
            if target_node is None:
                raise ValueError("target_node required when success=True")
            if target_node not in self._nodes:
                if target_num_candidates is None:
                    raise ValueError("target_num_candidates required for a new node")
                self._add_new_node(target_node, target_num_candidates, group2remaining_candidate_ids=group2remaining_candidate_ids)
            self._G[node].add((edge_idx, target_node))
            self._G_rev[target_node].add((edge_idx, node))

            if not self._nodes[node].has_open_group(self.active_group):
                self._close_node(node)
            if self._nodes[target_node].has_open_group(self.active_group):
                self._rebuild_distances()
            else:
                self._close_node(target_node)
                self._maybe_advance_group(target_node)
        else:
            if not self._nodes[node].has_open_group(self.active_group):
                self._close_node(node)
                self._maybe_advance_group(node)

    def get_distance(self, node: Hashable) -> Optional[int]:
        d = self._dist.get(node)
        return None if d is None or d == float("inf") else d

    def get_next_hop(self, node: Hashable) -> Optional[Hashable]:
        if node in self._frontier:
            return node
        nxt = self._next.get(node)
        if nxt is None:
            return None
        if isinstance(nxt, tuple) and len(nxt) == 2:
            return nxt[1]
        return nxt

    def edge_info(self, node: Hashable, edge_idx: Hashable) -> np.ndarray:
        return self._nodes[node].edge_data[edge_idx]

    def is_finished(self) -> bool:
        return not self._frontier

    @property
    def active_group(self) -> int:
        return self._active_group

    @property
    def empty(self) -> bool:
        return self._empty

    def _add_new_node(
        self,
        node: Hashable,
        n_candidates: int,
        group2remaining_candidate_ids: Optional[List[Set[int]]] = None,
    ) -> None:
        if n_candidates < 1:
            raise ValueError("num_candidates must be positive")
        self._nodes[node] = NodeInfo(node, n_candidates, self._n_groups, group2remaining_candidate_ids=group2remaining_candidate_ids)
        self._G[node] = set()
        self._G_rev[node] = set()
        if self._empty:
            self._empty = False
        if self._nodes[node].has_open_group(self.active_group):
            self._frontier.add(node)
        else:
            self._close_node(node)
            self._maybe_advance_group(node)

    def _close_node(self, node: Hashable) -> None:
        info = self._nodes[node]
        if info.closed:
            return
        info.closed = True
        self._frontier.discard(node)
        self._rebuild_distances()

    def _rebuild_distances(self) -> None:
        self._dist.clear()
        self._next.clear()
        dq = deque(self._frontier)
        for _, info in self._nodes.items():
            info.distance = INFINITY
            self._dist[info.name] = INFINITY
        for src in self._frontier:
            self._nodes[src].distance = 0
            self._dist[src] = 0
        while dq:
            v = dq.popleft()
            v_dist = self._dist.get(v, INFINITY)
            for edge_idx, u in self._G_rev.get(v, ()):
                u_info = self._nodes[u]
                u_dist = self._dist.get(u, INFINITY)
                u_info.edge_data["distance"][edge_idx] = v_dist + 1
                if u_dist > u_info.edge_data["distance"][edge_idx]:
                    u_info.distance = u_info.edge_data["distance"][edge_idx]
                    self._dist[u] = u_info.edge_data["distance"][edge_idx]
                    self._next[u] = (edge_idx, v)
                    dq.append(u)

    def _maybe_advance_group(self, current_node: Hashable) -> None:
        distance = self._nodes[current_node].distance
        while distance == INFINITY and self.active_group < self._n_groups - 1:
            self._active_group += 1
            self._dist.clear()
            self._next.clear()
            self._frontier.clear()
            for _, info in self._nodes.items():
                info.active_group = self.active_group
                if info.has_open_group(self.active_group):
                    self._frontier.add(info.name)
                    info.closed = False
            self._rebuild_distances()
            distance = self._dist.get(current_node)

    def choose_edge(self, node: Hashable, return_reasoning: bool = False):
        info = self._nodes[node]
        if info.has_open_group(self.active_group):
            untested: List[int] = []
            for gid in range(self.active_group + 1):
                untested.extend(info.group2remaining_candidate_ids[gid])
            if not untested:
                raise ValueError("no untested edges in active group while group reports open")
            edge_idx = random.choice(untested)
            reasoning = f"random untested edge {edge_idx} from group<={self.active_group}"
        else:
            lowest_dist = info.distance
            candidates = [
                i for i, d in enumerate(info.edge_data)
                if d["distance"] <= lowest_dist and d["result"] == 1 and d["group"] <= self.active_group
            ]
            edge_idx = random.choice(candidates)
            reasoning = f"BFS-next edge {edge_idx} at dist {lowest_dist}"
        if return_reasoning:
            return edge_idx, reasoning
        return edge_idx


In [ ]:
import hashlib
from collections import deque
from typing import Dict, List, Optional, Tuple

import numpy as np


class FrameProcessor:
    OFFSETS4: Tuple[Tuple[int, int], ...] = ((-1, 0), (1, 0), (0, -1), (0, 1))
    OFFSETS8: Tuple[Tuple[int, int], ...] = (
        (-1, -1), (-1, 1), (1, -1), (1, 1), (-1, 0), (1, 0), (0, -1), (0, 1),
    )

    def __init__(self) -> None:
        self.connectivity_rank = 4
        self.status_bar_mode = "rule"
        self.status_bar_distance_threshold = 3
        self.status_bar_ratio_threshold = 5
        self.status_bar_twins_threshold = 3
        self.frame_shape = (64, 64)
        self.status_bar_color = 16
        self.minimal_width = 2
        self.maximal_width = 32
        self.non_salient_color = {0, 1, 2, 3, 4, 5}
        self.salient_color = {6, 7, 8, 9, 10, 11, 12, 13, 14, 15}

    def segment_frame(self, frame: np.ndarray) -> Tuple[np.ndarray, List[Dict]]:
        h, w = frame.shape
        label_map = np.zeros((h, w), dtype=int) - 1
        components: List[Dict] = []
        cid = -1
        offsets = self.OFFSETS4 if self.connectivity_rank == 4 else self.OFFSETS8

        for y in range(h):
            for x in range(w):
                if label_map[y, x] != -1:
                    continue
                cid += 1
                color = int(frame[y, x])
                q = deque([(y, x)])
                label_map[y, x] = cid
                min_x = max_x = x
                min_y = max_y = y
                area = 0
                while q:
                    cy, cx = q.popleft()
                    area += 1
                    min_x, max_x = min(min_x, cx), max(max_x, cx)
                    min_y, max_y = min(min_y, cy), max(max_y, cy)
                    for dy, dx in offsets:
                        ny, nx = cy + dy, cx + dx
                        if (
                            0 <= ny < h and 0 <= nx < w
                            and label_map[ny, nx] == -1
                            and frame[ny, nx] == color
                        ):
                            label_map[ny, nx] = cid
                            q.append((ny, nx))
                rect_area = (max_x - min_x + 1) * (max_y - min_y + 1)
                components.append(dict(
                    bounding_box=(min_x, min_y, max_x, max_y),
                    color=color,
                    area=area,
                    is_rectangle=area == rect_area,
                ))

        for i, comp in enumerate(components):
            twins = [
                j for j, other in enumerate(components)
                if i != j
                and other["area"] == comp["area"]
                and other["is_rectangle"] == comp["is_rectangle"]
                and other["color"] == comp["color"]
            ]
            comp["number_of_twins"] = len(twins)
            comp["twin_ids"] = twins
        return label_map, components

    def identify_status_bars(
        self, segmented_frame: np.ndarray, frame_segments: List[Dict],
    ) -> Tuple[Optional[List[List[Dict]]], np.ndarray]:
        if self.status_bar_mode == "crude":
            return None, self._identify_status_bars_crude()
        if self.status_bar_mode == "rule":
            return self._identify_status_bars_with_rule(segmented_frame, frame_segments)
        raise ValueError(f"unsupported status bar mode: {self.status_bar_mode}")

    def _identify_status_bars_crude(self) -> np.ndarray:
        m = np.zeros(self.frame_shape, dtype=bool)
        t = self.status_bar_distance_threshold
        m[:t, :] = True
        m[-t:, :] = True
        m[:, :t] = True
        m[:, -t:] = True
        return m

    def _identify_status_bars_with_rule(
        self, segmented_frame: np.ndarray, frame_segments: List[Dict],
    ) -> Tuple[List[List[Dict]], np.ndarray]:
        checked: set = set()
        ids_list: List[List[int]] = []
        for i, segment in enumerate(frame_segments):
            if i in checked:
                continue
            checked.add(i)
            on_edges = self._check_segment_fully_on_edge(segment, edges=["any"])
            if not on_edges:
                continue
            directions = []
            if "left" in on_edges or "right" in on_edges:
                directions.append("vertical")
            if "top" in on_edges or "bottom" in on_edges:
                directions.append("horizontal")
            direction = "any" if len(directions) == 2 else directions[0]
            is_long = self._check_segment_ratio(segment, direction=direction)
            ids = [i]
            if not is_long:
                twin_ids = self._segment_twins_on_edge(segment, frame_segments)
                for tid in twin_ids:
                    checked.add(tid)
                if len(twin_ids) + 1 < self.status_bar_twins_threshold:
                    continue
                ids.extend(twin_ids)
            ids_list.append(ids)

        segments_list: List[List[Dict]] = []
        mask = np.zeros(segmented_frame.shape, dtype=bool)
        for ids in ids_list:
            group = []
            for sid in ids:
                mask[segmented_frame == sid] = True
                group.append(frame_segments[sid])
            segments_list.append(group)
        return segments_list, mask

    def _check_segment_fully_on_edge(self, segment: Dict, edges: Optional[List[str]] = None) -> List[str]:
        x1, y1, x2, y2 = segment["bounding_box"]
        edges = edges or ["any"]
        result: List[str] = []
        if "left" in edges or "any" in edges:
            if max(x1, x2) < self.status_bar_distance_threshold:
                result.append("left")
        if "right" in edges or "any" in edges:
            if min(x1, x2) > self.frame_shape[1] - self.status_bar_distance_threshold:
                result.append("right")
        if "top" in edges or "any" in edges:
            if max(y1, y2) < self.status_bar_distance_threshold:
                result.append("top")
        if "bottom" in edges or "any" in edges:
            if min(y1, y2) > self.frame_shape[0] - self.status_bar_distance_threshold:
                result.append("bottom")
        return result

    def _check_segment_ratio(self, segment: Dict, direction: Optional[str] = None) -> bool:
        direction = direction or "any"
        x_len = segment["bounding_box"][2] - segment["bounding_box"][0] + 1
        y_len = segment["bounding_box"][3] - segment["bounding_box"][1] + 1
        ratio = x_len / y_len
        if ratio >= self.status_bar_ratio_threshold and direction in ("any", "horizontal"):
            return True
        if ratio <= 1 / self.status_bar_ratio_threshold and direction in ("any", "vertical"):
            return True
        return False

    def _segment_twins_on_edge(
        self, segment: Dict, frame_segments: List[Dict], edges: Optional[List[str]] = None,
    ) -> List[int]:
        if edges is None:
            edges = self._check_segment_fully_on_edge(segment, edges=["any"])
            if not edges:
                return []
        twins = []
        for tid in segment["twin_ids"]:
            if self._check_segment_fully_on_edge(frame_segments[tid], edges=edges):
                twins.append(tid)
        return twins

    @staticmethod
    def hash_frame(frame: np.ndarray) -> str:
        """Stable 128-bit hash for a 0-15 valued NumPy array; shape-aware."""
        frame = np.asarray(frame, dtype=np.uint8, order="C")
        flat = frame.ravel()
        if flat.size & 1:
            flat = np.concatenate([flat, np.zeros(1, dtype=np.uint8)])
        packed = (flat[0::2] << 4) | (flat[1::2] & 0x0F)
        return hashlib.blake2b(
            packed.tobytes(),
            digest_size=16,
            person=repr(frame.shape).encode(),
        ).hexdigest()

    def frame_segments_to_action_groups(
        self, frame_segments: List[Dict], n_groups: int,
    ) -> List[set]:
        """Bucket segments into priority groups by salience and size.

        Group 0 = salient + medium-size (most likely interactive UI).
        Group 1 = medium-size only.
        Group 2 = salient only.
        Group 3 = everything else except status bars.
        Group 4 = status bars.
        """
        if n_groups != 5:
            raise ValueError("only n_groups == 5 is currently supported")
        groups: List[set] = [set(), set(), set(), set(), set()]
        for sid, segment in enumerate(frame_segments):
            x_w = segment["bounding_box"][2] - segment["bounding_box"][0] + 1
            y_w = segment["bounding_box"][3] - segment["bounding_box"][1] + 1
            is_salient = segment["color"] in self.salient_color
            is_medium = self.minimal_width <= x_w <= self.maximal_width and self.minimal_width <= y_w <= self.maximal_width
            is_status_bar = segment["color"] == self.status_bar_color
            if is_salient and is_medium:
                groups[0].add(sid)
            elif is_medium:
                groups[1].add(sid)
            elif is_salient:
                groups[2].add(sid)
            elif not is_status_bar:
                groups[3].add(sid)
            else:
                groups[4].add(sid)
        return groups


In [ ]:
import logging
import random
import time
from typing import Any, Optional

import numpy as np
from arcengine import FrameData, GameAction, GameState


logger = logging.getLogger(__name__)


SIMPLE_ACTION_ID2GAME_ACTION = {
    1: GameAction.ACTION1,
    2: GameAction.ACTION2,
    3: GameAction.ACTION3,
    4: GameAction.ACTION4,
    5: GameAction.ACTION5,
}


class GraphAgent(Agent):
    """Frame-graph exploration with priority-group ordering on action candidates."""

    MAX_ACTIONS: int = 1_000_000
    N_GROUPS: int = 5
    TOTAL_TIME_BUDGET_S: float = 7.9 * 60 * 60

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        random.seed(int(time.time() * 1e6) ^ (hash(self.game_id) & 0xFFFFFFFF))

        self.frame_processor = FrameProcessor()
        self.graph_explorer = GraphExplorer(n_groups=self.N_GROUPS)

        self.status_bar_mask: Optional[np.ndarray] = None
        self.hashed_frame2action_results: dict = {}
        self.hashed_frame2transitions: dict = {}

        self.level_first_frame: Optional[str] = None
        self.last_hashed_frame: Optional[str] = None
        self.last_action: Optional[int] = None
        self.last_action_object: GameAction = GameAction.RESET
        self.last_levels_completed: int = 0
        self.level_up: bool = True
        self.failed: bool = False
        self.last_transition_suspicious: bool = False

        self.time_start = time.time()

    @property
    def name(self) -> str:
        return f"{super().name}.{self.MAX_ACTIONS}"

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def _get_frame_buffers(self, hashed_frame: str, num_actions: int):
        results = self.hashed_frame2action_results.setdefault(hashed_frame, np.zeros(num_actions))
        transitions = self.hashed_frame2transitions.setdefault(hashed_frame, [0] * num_actions)
        return results, transitions

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state is GameState.NOT_PLAYED:
            self.last_hashed_frame = None
            self.last_action = None
            if self.failed:
                self.level_up = True
                self.failed = False
            return GameAction.RESET

        if latest_frame.state is GameState.GAME_OVER:
            self.last_transition_suspicious = True
            return GameAction.RESET

        cur_levels = int(latest_frame.levels_completed)
        if cur_levels > self.last_levels_completed:
            self.level_up = True
            self.status_bar_mask = None
        self.last_levels_completed = cur_levels

        latest_np = np.array(latest_frame.frame, dtype=np.uint8)
        if latest_np.size == 0:
            return self._random_fallback(latest_frame)
        num_frames = latest_np.shape[0]
        latest_np = latest_np[-1]

        if self.level_up:
            seg_for_sb, segs_for_sb = self.frame_processor.segment_frame(latest_np)
            _, mask = self.frame_processor.identify_status_bars(seg_for_sb, segs_for_sb)
            self.status_bar_mask = mask
            self.hashed_frame2action_results = {}
            self.hashed_frame2transitions = {}

        latest_np[self.status_bar_mask] = 16
        segmented_frame, frame_segments = self.frame_processor.segment_frame(latest_np)
        available = list(latest_frame.available_actions or [])

        num_click_actions = 0
        num_actions = 0
        arrow_actions: list[GameAction] = []

        if 6 in available:
            num_click_actions = len(frame_segments)
            num_actions += num_click_actions
            action_groups = self.frame_processor.frame_segments_to_action_groups(
                frame_segments, n_groups=self.N_GROUPS
            )
        else:
            action_groups = [set() for _ in range(self.N_GROUPS)]

        for aid in available:
            if aid in SIMPLE_ACTION_ID2GAME_ACTION:
                arrow_actions.append(SIMPLE_ACTION_ID2GAME_ACTION[aid])
                action_groups[0].add(num_actions)
                num_actions += 1

        latest_np[latest_np == 16] = 0
        hashed_frame = self.frame_processor.hash_frame(latest_np)

        if self.level_up:
            self.level_first_frame = hashed_frame
            self.graph_explorer.reset()
            self.graph_explorer.initialize(
                start_node=hashed_frame,
                num_candidates=num_actions,
                group2remaining_candidate_ids=action_groups,
            )
            self.level_up = False

        if self.last_hashed_frame is not None:
            transition = hashed_frame != self.last_hashed_frame
            suspicious = (hashed_frame == self.level_first_frame and num_frames > 1) or self.last_transition_suspicious
            self.last_transition_suspicious = False

            prev_results, prev_transitions = self._get_frame_buffers(
                self.last_hashed_frame,
                len(self.hashed_frame2action_results[self.last_hashed_frame]),
            )
            if transition:
                prev_results[self.last_action] = 1
                prev_transitions[self.last_action] = hashed_frame
            else:
                prev_results[self.last_action] = -1
                prev_transitions[self.last_action] = None

            try:
                self.graph_explorer.record_test(
                    self.last_hashed_frame,
                    self.last_action,
                    int(transition),
                    hashed_frame,
                    target_num_candidates=num_actions,
                    group2remaining_candidate_ids=action_groups,
                    suspicious_transition=suspicious,
                )
            except KeyError:
                logger.warning("graph_explorer lost prior node; resetting from current frame")
                self.graph_explorer.reset()
                self.graph_explorer.initialize(
                    start_node=hashed_frame,
                    num_candidates=num_actions,
                    group2remaining_candidate_ids=action_groups,
                )

        cur_results, _ = self._get_frame_buffers(hashed_frame, num_actions)
        available_mask = np.where(cur_results != -1)[0]
        if len(available_mask) == 0:
            return self._random_fallback(latest_frame)

        if hashed_frame in self.graph_explorer._nodes:
            action_id = self.graph_explorer.choose_edge(hashed_frame)
        else:
            action_id = int(random.choice(available_mask))

        arrow_control = action_id >= num_click_actions

        if arrow_control:
            action = arrow_actions[action_id - num_click_actions]
            action.reasoning = {"desired_action": str(action.value), "my_reason": "arrow"}
        else:
            mask = segmented_frame == action_id
            pts = np.argwhere(mask)
            y, x = pts[random.randint(0, len(pts) - 1)]
            action = GameAction.ACTION6
            action.set_data({"x": int(x), "y": int(y)})
            action.reasoning = {"desired_action": str(action.value), "my_reason": f"click seg {action_id}"}

        self.last_hashed_frame = hashed_frame
        self.last_action = action_id
        self.last_action_object = action
        return action

    def _random_fallback(self, latest_frame: FrameData) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET
        action = random.choice([a for a in GameAction if a is not GameAction.RESET])
        if action.is_complex():
            action.set_data({"x": random.randint(0, 63), "y": random.randint(0, 63)})
            action.reasoning = {"desired_action": str(action.value), "my_reason": "fallback"}
        else:
            action.reasoning = "fallback"
        return action

    def main(self) -> None:
        self.timer = time.time()
        while (
            not self.is_done(self.frames, self.frames[-1])
            and self.action_counter <= self.MAX_ACTIONS
        ):
            try:
                action = self.choose_action(self.frames, self.frames[-1])
            except Exception:
                logger.exception("choose_action crashed; recovering with last action")
                self.failed = True
                self.level_up = True
                action = self.last_action_object
            frame = self._step(action)
            if frame is not None:
                self._append(frame)
            self.action_counter += 1
            if time.time() - self.time_start > self.TOTAL_TIME_BUDGET_S:
                logger.info("time budget exhausted for %s", self.game_id)
                break


## 4. Run graph agent across all games

In [ ]:
import json, time
from pathlib import Path

ENV_DIR = Path(os.environ["ENVIRONMENTS_DIR"])
games = sorted({p.name for p in ENV_DIR.iterdir()
                if p.is_dir() and any(level.is_dir() for level in p.iterdir())})
print(f"Discovered {len(games)} games")

# Per-game wall-clock budget. Kaggle ceiling = 9h; reserve 1h for setup/teardown.
TOTAL_BUDGET_S = 8.0 * 60 * 60
PER_GAME_S = TOTAL_BUDGET_S / max(len(games), 1)
GraphAgent.MAX_ACTIONS = 1_000_000
GraphAgent.TOTAL_TIME_BUDGET_S = PER_GAME_S

arc = Arcade()
card_id = arc.open_scorecard(tags=["graph", "v1"])
log.info(f"scorecard {card_id}, mode {arc.operation_mode}, per-game budget {PER_GAME_S/60:.1f} min")

t0 = time.time()
for gid in games:
    try:
        env = arc.make(gid, scorecard_id=card_id)
        if env is None:
            log.warning("env is None for %s; skipping", gid); continue
        agent = GraphAgent(card_id=card_id, game_id=gid, agent_name="graph", arc_env=env, tags=["graph", "v1"])
        agent.main()
        log.info(f"{gid}: levels {agent.frames[-1].levels_completed}, actions {agent.action_counter}, total {time.time()-t0:.0f}s")
    except Exception:
        log.exception(f"{gid} crashed; continuing")

scorecard = arc.close_scorecard(card_id)
dumped = scorecard.model_dump()
print(f"=== Final: {dumped['total_levels_completed']}/{dumped['total_levels']} levels, "
      f"{dumped['total_environments_completed']}/{dumped['total_environments']} envs, "
      f"{dumped['total_actions']} actions ===")


## 5. Dump scorecard

In [ ]:
with open("/kaggle/working/scorecard.json", "w") as f:
    json.dump(dumped, f, indent=2, default=str)
print("wrote /kaggle/working/scorecard.json")
for env in dumped["environments"]:
    run = env["runs"][0] if env["runs"] else {}
    print(f"{env['id']:>22} | levels {env['levels_completed']:>2}/{env['level_count']:>2} | actions {env['actions']:>5} | resets {env['resets']}")
